# 1. Install dependencies
Install LangGraph and the OpenAI integration.

In [ ]:
!pip -q install langgraph langchain-openai

# 2. Load the API key from Colab Secrets

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 3. Imports and model name

In [ ]:
from typing import Literal, TypedDict
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel
MODEL_NAME = "gpt-6-luna"

# 4. State and structured category

In [ ]:
class TicketState(TypedDict, total=False):
    ticket: str
    category: Literal["billing", "technical", "general"]
    team: str
    reply: str

class TicketCategory(BaseModel):
    category: Literal["billing", "technical", "general"]

# 5. Classifier setup

In [ ]:
classifier = ChatOpenAI(model=MODEL_NAME, temperature=0).with_structured_output(TicketCategory)

# 6. Classify a ticket
This is the only node that calls the LLM.

In [ ]:
def classify(state: TicketState) -> dict[str, str]:
    system_instruction = (
        "You are a support ticket classifier for TaskNest, a to-do app.\n"
        "Classify the ticket into exactly one category:\n"
        "- billing: payments, charges, invoices, refunds, subscriptions.\n"
        "- technical: bugs, crashes, error messages, login problems.\n"
        "- general: feature ideas, general questions, thank-you notes, anything else.\n"
        "Decide by the customer's main problem, not by single words. If something is broken "
        "(for example, the app crashes while paying), it is technical. If money was charged "
        "wrongly, it is billing."
    )
    decision = classifier.invoke([SystemMessage(content=system_instruction), HumanMessage(content=state["ticket"])])  # LLM call
    return {"category": decision.category}

# 7. Routing function

In [ ]:
def route(state: TicketState) -> Literal["billing", "technical", "general"]:
    return state["category"]  # LLM decision chooses the edge

# 8. Team nodes

In [ ]:
def billing_team(state: TicketState) -> dict[str, str]:
    return {"team": "Billing", "reply": "Thanks for reaching out! Your ticket was forwarded to our Billing team."}
def tech_team(state: TicketState) -> dict[str, str]:
    return {"team": "Tech", "reply": "Thanks for reaching out! Your ticket was forwarded to our Tech team."}
def general_team(state: TicketState) -> dict[str, str]:
    return {"team": "General", "reply": "Thanks for reaching out! Your ticket was forwarded to our General support team."}

# 9. Build and compile the graph

In [ ]:
builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing_team", billing_team)
builder.add_node("tech_team", tech_team)
builder.add_node("general_team", general_team)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, {"billing": "billing_team", "technical": "tech_team", "general": "general_team"})
builder.add_edge("billing_team", END)
builder.add_edge("tech_team", END)
builder.add_edge("general_team", END)
graph = builder.compile()

# 10. Draw the graph

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

# 11. One example run

In [ ]:
result = graph.invoke({"ticket": "I was charged twice this month"})
print(result)

# 12. Test tickets
Each ticket is invoked separately. Mismatches are printed without stopping. Assertions check output fields and every route.

In [ ]:
required_keys = {"ticket", "category", "team", "reply"}

ticket_1 = "I was charged twice this month"
result_1 = graph.invoke({"ticket": ticket_1})
expected_1 = "billing"
assert required_keys.issubset(result_1)
print("Ticket:", ticket_1, "| Expected:", expected_1, "| Chosen:", result_1["category"], "| Reply:", result_1["reply"], "|", "OK" if result_1["category"] == expected_1 else "MISMATCH")

ticket_2 = "How do I get a refund for my annual subscription?"
result_2 = graph.invoke({"ticket": ticket_2})
expected_2 = "billing"
assert required_keys.issubset(result_2)
print("Ticket:", ticket_2, "| Expected:", expected_2, "| Chosen:", result_2["category"], "| Reply:", result_2["reply"], "|", "OK" if result_2["category"] == expected_2 else "MISMATCH")

ticket_3 = "The app crashes every time I open a shared list"
result_3 = graph.invoke({"ticket": ticket_3})
expected_3 = "technical"
assert required_keys.issubset(result_3)
print("Ticket:", ticket_3, "| Expected:", expected_3, "| Chosen:", result_3["category"], "| Reply:", result_3["reply"], "|", "OK" if result_3["category"] == expected_3 else "MISMATCH")

ticket_4 = "I can't log in, the password reset email never arrives"
result_4 = graph.invoke({"ticket": ticket_4})
expected_4 = "technical"
assert required_keys.issubset(result_4)
print("Ticket:", ticket_4, "| Expected:", expected_4, "| Chosen:", result_4["category"], "| Reply:", result_4["reply"], "|", "OK" if result_4["category"] == expected_4 else "MISMATCH")

ticket_5 = "Could you add a dark mode?"
result_5 = graph.invoke({"ticket": ticket_5})
expected_5 = "general"
assert required_keys.issubset(result_5)
print("Ticket:", ticket_5, "| Expected:", expected_5, "| Chosen:", result_5["category"], "| Reply:", result_5["reply"], "|", "OK" if result_5["category"] == expected_5 else "MISMATCH")

ticket_6 = "Just wanted to say thank you, TaskNest is great!"
result_6 = graph.invoke({"ticket": ticket_6})
expected_6 = "general"
assert required_keys.issubset(result_6)
print("Ticket:", ticket_6, "| Expected:", expected_6, "| Chosen:", result_6["category"], "| Reply:", result_6["reply"], "|", "OK" if result_6["category"] == expected_6 else "MISMATCH")

# Ambiguous ticket: the app crashes during payment, so classify the main problem.
ticket_7 = "The app crashes every time I try to pay"
result_7 = graph.invoke({"ticket": ticket_7})
expected_7 = "technical"
assert required_keys.issubset(result_7)
print("Ticket:", ticket_7, "| Expected:", expected_7, "| Chosen:", result_7["category"], "| Reply:", result_7["reply"], "|", "OK" if result_7["category"] == expected_7 else "MISMATCH")

# Ambiguous ticket: payment succeeded but paid features are missing.
ticket_8 = "I paid for Premium but the Premium features don't show up"
result_8 = graph.invoke({"ticket": ticket_8})
expected_8 = "billing"
assert required_keys.issubset(result_8)
print("Ticket:", ticket_8, "| Expected:", expected_8, "| Chosen:", result_8["category"], "| Reply:", result_8["reply"], "|", "OK" if result_8["category"] == expected_8 else "MISMATCH")

# Ambiguous ticket: a discount question is about pricing.
ticket_9 = "Do you offer a student discount?"
result_9 = graph.invoke({"ticket": ticket_9})
expected_9 = "billing"
assert required_keys.issubset(result_9)
print("Ticket:", ticket_9, "| Expected:", expected_9, "| Chosen:", result_9["category"], "| Reply:", result_9["reply"], "|", "OK" if result_9["category"] == expected_9 else "MISMATCH")

# Ambiguous ticket: the customer is upset and requests a refund.
ticket_10 = "Your app is useless and I want my money back"
result_10 = graph.invoke({"ticket": ticket_10})
expected_10 = "billing"
assert required_keys.issubset(result_10)
print("Ticket:", ticket_10, "| Expected:", expected_10, "| Chosen:", result_10["category"], "| Reply:", result_10["reply"], "|", "OK" if result_10["category"] == expected_10 else "MISMATCH")

# Ambiguous ticket: the login problem prevents subscription cancellation.
ticket_11 = "I can't log in, so I can't cancel my subscription"
result_11 = graph.invoke({"ticket": ticket_11})
expected_11 = "technical"
assert required_keys.issubset(result_11)
print("Ticket:", ticket_11, "| Expected:", expected_11, "| Chosen:", result_11["category"], "| Reply:", result_11["reply"], "|", "OK" if result_11["category"] == expected_11 else "MISMATCH")

assert {result_1["category"], result_2["category"], result_3["category"], result_4["category"], result_5["category"], result_6["category"], result_7["category"], result_8["category"], result_9["category"], result_10["category"], result_11["category"]} == {"billing", "technical", "general"}, "Every team branch must be reached at least once."